# Kapitel 4

## Frågor

### 1. Vad kännetecknar klassificeringsproblem? Ge några exempel på tillämpningsområden.

In a classification problem the target y is a category (class), not a number. It can be binary (two classes, like yes/no) or multiclass (more than two classes).

Examples: spam detection, medical diagnosis (disease/no disease), image recognition (which digit, cat or dog), predicting whether a customer will leave, credit approval, and sentiment analysis (positive/negative/neutral).

### 2. Förklara hur OvR- och OvO-algoritmerna fungerar.

Both are ways to use binary classifiers for a multiclass problem.

- **OvR (One-vs-Rest)**: train one binary classifier per class, each separating "this class" from all the others. With 3 classes that's 3 classifiers. To predict, run all of them and pick the class whose classifier gives the highest score.
- **OvO (One-vs-One)**: train one binary classifier for every pair of classes. With 3 classes that's 3 pairs (A vs B, A vs C, B vs C), with N classes N·(N-1)/2. To predict, run all of them and pick the class that wins the most pairs.

OvR needs far fewer classifiers. OvO trains each classifier on only two classes' data, which can be faster per classifier and suits algorithms that scale badly with dataset size, like SVMs.

### 3. Förklara följande utvärderingsmått:

#### a) Confusion matrix

A table of actual classes against predicted classes, showing what the model got right and what kind of mistakes it made. For binary classification it has four cells:
- True Positives (TP): actually positive, predicted positive
- True Negatives (TN): actually negative, predicted negative
- False Positives (FP): actually negative, predicted positive (false alarm)
- False Negatives (FN): actually positive, predicted negative (miss)

The metrics below are calculated from these four numbers.

#### b) Accuracy

The share of all predictions that were correct: (TP + TN) / (TP + TN + FP + FN).

Easy to understand but misleading with imbalanced classes. If 95% of emails aren't spam, a model that always predicts "not spam" gets 95% accuracy without having learned anything.

#### c) Precision

Of everything the model predicted as positive, how many actually were positive: TP / (TP + FP).

High precision means few false alarms. It matters when false positives are costly, like marking a real email as spam.

#### d) Recall

Of all actual positives, how many the model found: TP / (TP + FN).

High recall means few misses. It matters when missing a positive is costly, like missing a cancer diagnosis.

#### e) F1-score

The harmonic mean of precision and recall: F1 = 2 · (precision · recall) / (precision + recall).

A single number that balances precision and recall, useful with imbalanced classes where accuracy is misleading. The harmonic mean is pulled down a lot if either precision or recall is low, so a model can't get a good F1 by maximizing only one of them.

#### f) ROC-kurvan

The ROC curve plots the true positive rate (recall) against the false positive rate (FP / (FP + TN)) for all classification thresholds. It shows the tradeoff between finding more positives and getting more false alarms.

A perfect classifier goes up to the top-left corner, random guessing gives a diagonal line. The area under the curve (ROC AUC) summarizes it in one number between 0 and 1, where 0.5 is random guessing and 1.0 is perfect. It's useful for comparing classifiers independently of which threshold is used.

### 4. Vad är precision-recall tradeoff för något?

Precision and recall usually move in opposite directions when you change the classification threshold of one model. Raising the threshold (the model has to be more confident before predicting positive) can never increase recall, since fewer rows get predicted positive. Precision usually goes up, but not necessarily at every single step. Lowering the threshold does the opposite.

So for a fixed model you pick the point on that curve that fits the use case, depending on whether false positives or false negatives cost more. A better model can improve both at the same time, so the tradeoff is about the threshold, not a rule that the two numbers can never both go up.

### 5. Några vanligt förekommande modeller för klassificeringsproblem är enligt nedan. Förklara översiktligt hur respektive modell fungerar. Läs även igenom respektive modells dokumentation, notera att du inte behöver förstå alla detaljer från dokumentationen men det är bra att ha läst igenom den.

#### a) Logistisk regression

Despite the name it's used for classification. It computes a weighted sum of the features, like linear regression, and passes it through the sigmoid function, which gives a value between 0 and 1 that is interpreted as a probability. If the probability is above a threshold (usually 0.5) it predicts class 1, otherwise class 0. The weights are learned by minimizing log loss (cross-entropy). It's simple, fast and interpretable, and works best when the classes are roughly linearly separable.

#### b) Support vector machines

SVC looks for the decision boundary (hyperplane) that separates the classes with the largest possible margin, the widest "street" between the closest points of each class (the support vectors). Only points close to the boundary affect it. With kernels like RBF it can separate data that isn't linearly separable, by working as if the data were mapped to a higher-dimensional space (the kernel trick). It works well on small and medium datasets, needs tuning of C and the kernel parameters, and gets slow on large datasets.

#### c) Beslutsträd

Like a regression tree but for classification. The data is split repeatedly on feature values, choosing splits that make each group as pure as possible (mostly one class), usually measured with Gini impurity or entropy. The prediction in a leaf is the most common class among the training rows there. Easy to interpret, handles non-linear boundaries, but overfits easily without limits like `max_depth`.

#### d) Ensemble learning

Combines several classifiers to get a better result than any single one.

- **VotingClassifier**: trains different types of models and combines their predictions, either by majority vote (hard voting) or by averaging predicted probabilities (soft voting, often better if the probabilities are well calibrated).
- **BaggingClassifier**: trains the same type of model several times on different bootstrap samples of the training data and combines the predictions by vote. Reduces variance, good for models that overfit easily like decision trees.

#### e) Random forest

Bagging with decision trees, where each split also only considers a random subset of the features. This makes the trees less correlated, so averaging them reduces variance more. It usually performs well with little tuning and overfits less than a single tree, but is harder to interpret (feature importances help somewhat).

#### f) Extra trees

Extremely Randomized Trees. Similar to random forest, but with more randomness in how splits are chosen. A normal tree searches for the best threshold for each feature it considers. Extra Trees instead draws one random threshold for each considered feature, and then picks the best of those random candidate splits. So there is still a comparison, just between random candidates instead of all possible thresholds. By default it also trains each tree on the whole dataset instead of a bootstrap sample.

Each tree gets a bit worse (more bias) but the trees become less correlated (less variance when averaged), and training is faster since it doesn't search every threshold. Performance is often close to random forest, sometimes better, sometimes worse.

### 6. Vad innebär det att vi kan kolla på feature importance med hjälp av trädmodeller såsom beslutsträd eller random forest?

Tree models can measure how much each feature reduced impurity over all the splits where it was used. Summed and normalized, this gives a relative importance score per feature, where higher means the model relied more on that feature.

It's useful for explaining which features the model uses, for feature selection (dropping features with near-zero importance), and as a sanity check that the model uses sensible features rather than some artifact in the data. It doesn't show direction (whether high values increase or decrease the prediction) or causality, and correlated features share importance between them, so each can look less important than it is.

### 7. Stina säger till Kalle på lunchsamtalet “jag vill ha högsta möjliga precision för vår klassificieringsmodell”. Kalle funderar ett tag och säger “men vad händer då med recall”? Vad hade du svarat? I vilka fall kan man tänka sig vilja ha en så hög precision som möjligt? I vilka fall kan det vara dåligt? Om vi tänker oss rättsväsendet där en slutgiltig dom kan leda till fängelse, vad kan vi då säga om precision-recall tradeoff?

Kalle has a point. If Stina pushes precision as high as possible, for example by only predicting positive when the model is extremely confident, recall will usually drop a lot, since many actual positives won't reach that threshold. For one fixed model, raising the threshold can never increase recall, while precision usually but not always goes up. A genuinely better model could improve both, so the tradeoff is about where the threshold is set, not a law.

When very high precision makes sense: when a false positive is costly and missing some positives is acceptable. For example a spam filter, where deleting a real important email is worse than letting some spam through.

When it's bad: when missing a positive is the dangerous outcome, like disease screening. Optimizing only for precision there means missing sick patients (false negatives), which is usually worse than some extra follow-up tests (false positives).

In the justice system a "positive" would be "guilty". High precision means that when the system says guilty it is very likely correct, so few innocent people are convicted, but recall drops, so more guilty people go free. Since wrongly imprisoning an innocent person is generally seen as much worse than letting a guilty person go ("better that ten guilty persons escape than that one innocent suffer"), the justice system should deliberately favor precision over recall.

### 8. Förklara hur man kan tolka figur 4.8 på sidan 175.

Figure 4.8 shows the decision boundary of a logistic regression model that classifies points as class 0 (purple) or class 1 (yellow) based on x1 and x2. The background color shows the predicted probability of class 1 (colorbar from 0 to 1): purple areas have low probability, yellow areas high probability, and the green band in between is where the model is uncertain (close to 0.5).

The data points are plotted on top in their true class colors. Most purple points are in the purple area and most yellow in the yellow area, with some points on the wrong side of the boundary.

The boundary is a straight line, since logistic regression is a linear model: the line where the probability is 0.5 is always straight (a flat hyperplane with more features), it can't bend around groups of points the way a tree or a kernel SVM can. It also connects to question 4: moving the threshold away from 0.5 gives a different, parallel line as decision boundary, which changes the balance between precision and recall.

### 9. På sidan 209 står det “på träningsdatan använder vi .fit_transform(), på valideringsdatan och testdatan använder vi endast .transform().” Förklara logiken bakom detta.

`.fit()` learns the parameters of the transformation (e.g. mean and standard deviation for StandardScaler) from the data it gets. `.transform()` applies the transformation using parameters that were already learned.

We learn these parameters only from the training data, and apply the same transformation to validation and test data. If we used `.fit_transform()` on validation or test data, they would be scaled with their own mean and standard deviation, so they would end up on a different scale than the one the model was trained on (a value of 1.5 would mean different things), and the predictions would be wrong. It would also let information from the validation/test data into the preprocessing, which is the same kind of leakage as using test data in any other modeling decision. In production, new data has to be transformed with the same training-based parameters for the same reason.

### 11. Förklara vad nedanstående kod gör och tolka resultatet från koden.

In [1]:
from sklearn.metrics import classification_report

y_true = [0, 1, 2, 2, 2]
y_pred = [0, 0, 2, 2, 1]
target_names = ['class 0', 'class 1', 'class 2']
print(classification_report(y_true, y_pred, target_names=target_names))

              precision    recall  f1-score   support

     class 0       0.50      1.00      0.67         1
     class 1       0.00      0.00      0.00         1
     class 2       1.00      0.67      0.80         3

    accuracy                           0.60         5
   macro avg       0.50      0.56      0.49         5
weighted avg       0.70      0.60      0.61         5



The code prints precision, recall, f1-score and support (number of actual samples) for each class, plus overall accuracy and two averages.

Class 0: there is 1 actual class 0 sample and it was predicted correctly, so recall = 1.00. But the model predicted class 0 twice in total (the other time for a sample that was actually class 1), so only 1 of 2 class 0 predictions was right, precision = 0.50.

Class 1: the only class 1 sample was predicted as class 0, and the one time the model predicted class 1 it was wrong (the true class was 2). So precision, recall and f1 are all 0.00.

Class 2: 3 actual samples, 2 predicted correctly and 1 predicted as class 1, so recall = 2/3 = 0.67. Every class 2 prediction was correct, so precision = 1.00.

Accuracy is 3/5 = 0.60. Macro avg is the plain average over the classes, weighted avg weights each class by its support. Weighted precision (0.70) is higher than macro precision (0.50) because class 2, which has perfect precision, has the most samples.

### 13. I föregående kapitel gjordes en EDA på datasetet “hr_employee_data.xlsx”. Gör nu ett komplett ML-flöde där den beroende variabeln, y, är left. Om den variabeln är 1 så betyder det att den anställde har lämnat företaget och om den är 0 så betyder det att den anställde inte har lämnat företaget, det vill säga jobbar kvar.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = pd.read_excel("data/hr_employee_data.xlsx")
df = df.drop(columns="Emp_Id")
df.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


Department is nominal and salary is ordinal (low < medium < high). I used dummy encoding for both to keep it simple. Tree models work fine with that, and so does logistic regression.

In [3]:
df = pd.get_dummies(df, columns=["Department", "salary"], drop_first=True)

X = df.drop(columns="left")
y = df["left"]

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

I used `stratify=y` in both splits since only about 24% of the employees left, so that the share stays the same in train, validation and test.

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

log_reg = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
log_pred = log_reg.predict(X_val_scaled)

rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_val)

print("=== Logistic Regression ===")
print(classification_report(y_val, log_pred))
print("=== Random Forest ===")
print(classification_report(y_val, rf_pred))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.83      0.93      0.88      1829
           1       0.64      0.39      0.48       571

    accuracy                           0.80      2400
   macro avg       0.73      0.66      0.68      2400
weighted avg       0.78      0.80      0.78      2400

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1829
           1       0.99      0.96      0.98       571

    accuracy                           0.99      2400
   macro avg       0.99      0.98      0.99      2400
weighted avg       0.99      0.99      0.99      2400



I scaled the features for logistic regression since they are on very different scales (satisfaction_level 0-1, average_montly_hours around 100-300), which helps it converge. Random forest doesn't need scaling since tree splits don't depend on feature scale.

Random forest did much better on this split: accuracy 0.99 vs 0.80. Logistic regression mainly struggled with class 1 (people who left), recall only 0.39. A likely reason is that the relationship between the features and leaving isn't linear (the satisfaction histogram in chapter 3 had several separate groups of leavers), and a tree model can handle that. The duplicate check further down affects how much to trust the 0.99 though.

In [6]:
print(confusion_matrix(y_val, rf_pred))
print("RF ROC AUC:", roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1]))
print("LogReg ROC AUC:", roc_auc_score(y_val, log_reg.predict_proba(X_val_scaled)[:, 1]))

[[1826    3]
 [  20  551]]
RF ROC AUC: 0.9949054874808375
LogReg ROC AUC: 0.8275123784062759


Random forest has 3 false positives and 20 false negatives out of 2400 validation rows, and ROC AUC 0.995 vs 0.828 for logistic regression. I continued with random forest.

In [7]:
hyper_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    hyper_params,
    scoring="f1",
    cv=3,
    n_jobs=-1,
)
grid.fit(X_train, y_train)
print(grid.best_params_)

tuned_pred = grid.best_estimator_.predict(X_val)
print(classification_report(y_val, tuned_pred))

{'max_depth': None, 'n_estimators': 200}
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1829
           1       1.00      0.97      0.98       571

    accuracy                           0.99      2400
   macro avg       0.99      0.98      0.99      2400
weighted avg       0.99      0.99      0.99      2400



Tuning changed very little compared to the default random forest.

In [8]:
final_model = RandomForestClassifier(**grid.best_params_, random_state=42)
final_model.fit(X_train_full, y_train_full)

test_pred = final_model.predict(X_test)
print(classification_report(y_test, test_pred))
print(confusion_matrix(y_test, test_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      2286
           1       1.00      0.96      0.98       714

    accuracy                           0.99      3000
   macro avg       0.99      0.98      0.99      3000
weighted avg       0.99      0.99      0.99      3000

[[2283    3]
 [  26  688]]


On the test set accuracy is 0.99 and recall for people who left is 0.96. Precision for class 1 is shown as 1.00 because of rounding, the confusion matrix shows 3 false positives. The duplicate check below affects how much to trust this result.

In [9]:
dupes = df.duplicated().sum()
print("duplicate rows (features + label):", dupes)

train_profiles = set(map(tuple, X_train.values.tolist()))
val_overlap = X_val.apply(lambda r: tuple(r) in train_profiles, axis=1).sum()
print(f"validation rows with a feature profile already in training: {val_overlap}/{len(X_val)}")

train_full_profiles = set(map(tuple, X_train_full.values.tolist()))
test_overlap = X_test.apply(lambda r: tuple(r) in train_full_profiles, axis=1).sum()
print(f"test rows with a feature profile already in train+val: {test_overlap}/{len(X_test)}")

duplicate rows (features + label): 3008
validation rows with a feature profile already in training: 644/2400
test rows with a feature profile already in train+val: 934/3000


About 3000 rows are exact duplicates of another row, and a big part of the validation and test rows have a feature profile that also exists in the training data. `Emp_Id` is unique, so I can't tell if these are different employees who happen to have identical rounded values, or duplicated records. Either way, those rows are easier for the model than a completely new profile.

To check how much this matters, I compared with a grouped split where identical feature profiles always end up on the same side.

In [10]:
from sklearn.model_selection import GroupShuffleSplit

profile = X.astype(str).agg("|".join, axis=1)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=profile))
print("shared profiles between the two sides:", len(set(profile.iloc[train_idx]) & set(profile.iloc[test_idx])))

grouped_model = RandomForestClassifier(**grid.best_params_, random_state=42)
grouped_model.fit(X.iloc[train_idx], y.iloc[train_idx])
grouped_pred = grouped_model.predict(X.iloc[test_idx])
print(classification_report(y.iloc[test_idx], grouped_pred, digits=3))
print(confusion_matrix(y.iloc[test_idx], grouped_pred))

shared profiles between the two sides: 0
              precision    recall  f1-score   support

           0      0.971     0.997     0.984      2270
           1      0.989     0.906     0.946       720

    accuracy                          0.975      2990
   macro avg      0.980     0.951     0.965      2990
weighted avg      0.975     0.975     0.975      2990

[[2263    7]
 [  68  652]]


With the grouped split accuracy was 0.975 instead of 0.990, and recall for people who left dropped from about 0.96 to about 0.91. So the overlap did make the first score look better than it should, but the model still does well on profiles it hasn't seen. The grouped number is the one I would report. I only did a single grouped split here, a grouped cross-validation (`GroupKFold`) would give a more stable estimate.

In [11]:
for f, imp in sorted(zip(X.columns, final_model.feature_importances_), key=lambda t: -t[1])[:8]:
    print(f, round(imp, 3))

satisfaction_level 0.317
time_spend_company 0.188
number_project 0.184
average_montly_hours 0.148
last_evaluation 0.122
Work_accident 0.01
salary_low 0.008
salary_medium 0.004


Satisfaction level has the highest importance, followed by time_spend_company, number_project, average_montly_hours and last_evaluation. Department and salary barely contribute.

This shows which features the model uses for its splits, not why people left. It fits with the idea from the chapter 3 EDA that there are different groups of leavers (for example unsatisfied people, and people with a heavy workload), but it doesn't prove it.

In [12]:
production_model = RandomForestClassifier(**grid.best_params_, random_state=42)
production_model.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

Last step: retrain on all data (train+val+test) with the chosen settings. This is the model that would be used. Training on more data doesn't fix the evaluation issue above, the grouped result is still the performance estimate I would trust.

### 15. I avsnitt 4.4.1 “Kodexempel 1 - Klassificering av MNIST” modelleras MNIST. Skriv kod så att du kan ta egna bilder (via exempelvis mobilen) på handskrivna siffror och prediktera dessa.

I have 5 images of handwritten digits in `kap4/images/`: three with one digit (5.png, 7.png, 3.png) and two with two digits (12.png, 48.png), drawn as black lines on white background. The main work is preprocessing: invert the colors, crop to the digit(s), split the two-digit images into separate digits, and scale each digit so it looks like an MNIST sample (28x28, centered, similar stroke thickness) before predicting.

In [13]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from PIL import Image, ImageFilter
import numpy as np

mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(np.uint8)

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X, y)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

In [14]:
def segment_digits(arr, gap=3):
    col_has_ink = (arr > 0).any(axis=0)
    segments, start, gap_count = [], None, 0
    for i, has in enumerate(col_has_ink):
        if has:
            if start is None:
                start = i
            gap_count = 0
        else:
            if start is not None:
                gap_count += 1
                if gap_count > gap:
                    segments.append((start, i - gap_count))
                    start = None
    if start is not None:
        segments.append((start, len(col_has_ink) - 1))
    return segments


def digit_to_mnist_format(sub_arr):
    rows = np.any(sub_arr > 0, axis=1)
    top, bottom = np.where(rows)[0][[0, -1]]
    cropped = sub_arr[top:bottom + 1, :]
    cols = np.any(cropped > 0, axis=0)
    left, right = np.where(cols)[0][[0, -1]]
    cropped = cropped[:, left:right + 1]

    h, w = cropped.shape
    scale = 20 / max(h, w)
    new_h, new_w = max(1, round(h * scale)), max(1, round(w * scale))
    digit_img = Image.fromarray(cropped).resize((new_w, new_h))

    digit_img = digit_img.filter(ImageFilter.MaxFilter(3))

    canvas = Image.new("L", (28, 28), 0)
    canvas.paste(digit_img, ((28 - new_w) // 2, (28 - new_h) // 2))
    canvas = canvas.filter(ImageFilter.GaussianBlur(0.5))
    return np.array(canvas).reshape(1, -1).astype("float64")


def preprocess_and_predict(path):
    img = Image.open(path).convert("L")
    arr = np.array(img)
    arr = 255 - arr
    arr[arr < 80] = 0

    segments = segment_digits(arr)
    digits = []
    for c1, c2 in segments:
        digit_arr = digit_to_mnist_format(arr[:, c1:c2 + 1])
        digits.append(str(clf.predict(digit_arr)[0]))
    return "".join(digits)


for fname, expected in [("images/5.png", "5"), ("images/7.png", "7"),
                         ("images/3.png", "3"), ("images/12.png", "12"),
                         ("images/48.png", "48")]:
    predicted = preprocess_and_predict(fname)
    print(f"{fname}: expected={expected} predicted={predicted}")

images/5.png: expected=5 predicted=3
images/7.png: expected=7 predicted=2
images/3.png: expected=3 predicted=3
images/12.png: expected=12 predicted=12
images/48.png: expected=48 predicted=48


3 out of 5 images were completely correct (3, 12 and 48). Counting each digit separately, 5 out of 7 were correct. 5 was read as 3, and 7 as 2.

The preprocessing inverts the colors, crops each digit, keeps the aspect ratio when scaling it into a 20x20 box, centers it on a 28x28 canvas, and thickens the strokes since my lines are much thinner than MNIST's. I would need to compare the processed images with real MNIST digits before saying why 5 and 7 were misread. This shows what the book means by preprocessing not being trivial.

I skipped the Streamlit app since it was optional.